In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# If running on Kaggle, this is fine to run as-is.
!pip -q install pulp requests pandas tabulate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 65.9 MB/s eta 0:00:00:00:0100:01


In [3]:
from __future__ import annotations
import math
import sys
from typing import Dict, List, Optional, Set, Tuple

import requests
import pandas as pd
from tabulate import tabulate

try:
    import pulp  # ILP solver
except Exception:
    print("This notebook requires PuLP. Please run the install cell above.", file=sys.stderr)
    raise

# ----- Configuration (edit these) -----
BUDGET_M = 100.0            # Budget in £m
ALLOW_FLAGGED = True       # include flagged/unavailable players if True
LOCK = {"Haaland"}          # force-include names (substring match, case-insensitive)
BAN = set()                 # force-exclude names (substring match)
MIN_FROM_TEAM = {}          # e.g., {"Manchester City": 2}
VERBOSE_SOLVER = False      # True to print solver logs

# ==== HORIZON / FDR CONFIG ====
HORIZON_GWS = 4            # plan over the next K gameweeks (edit as you like)
USE_FDR_FOR_GW1 = False    # if True, also weight the *next* GW by fixture difficulty (ep_next may already include it)

# Map FPL difficulty (1 easiest … 5 hardest) -> multiplier.
# Tweak these to your taste; keep the mean near ~1.0.
DIFF_TO_MULT = {
    1: 1.20,   # very easy
    2: 1.10,
    3: 1.00,
    4: 0.85,
    5: 0.70,   # very hard
}




In [4]:
FPL_API_BASE = "https://fantasy.premierleague.com/api"
BOOTSTRAP_URL = f"{FPL_API_BASE}/bootstrap-static/"  # public, no auth

# Squad rules
REQUIRED_COUNTS = {"GK": 2, "DEF": 5, "MID": 5, "FWD": 3}
MAX_PER_TEAM = 3

def fetch_bootstrap() -> dict:
    r = requests.get(BOOTSTRAP_URL, timeout=30)
    r.raise_for_status()
    return r.json()






In [5]:
# ---- New config ----
MIN_PLAY_CHANCE = 0   # set 0 to include everyone; 25 to include 'doubtful+'; 75 to mimic old

def build_player_table(bootstrap: dict, allow_flagged: bool = False, min_play_chance: Optional[int] = None) -> pd.DataFrame:
    import math
    elements = pd.DataFrame(bootstrap["elements"])
    teams = pd.DataFrame(bootstrap["teams"])

    team_id_to_name = teams.set_index("id")["name"].to_dict()
    id_to_pos = {1: "GK", 2: "DEF", 3: "MID", 4: "FWD"}

    df = elements[[
        "id","web_name","team","element_type","now_cost",
        "ep_next","points_per_game","status","chance_of_playing_next_round"
    ]].copy()

    df["team_id"]   = df["team"]
    df["team_name"] = df["team_id"].map(team_id_to_name)
    df["pos_code"]  = df["element_type"].map(id_to_pos)
    df.rename(columns={"web_name":"name"}, inplace=True)

    def to_float(x):
        try: return float(x)
        except Exception: return math.nan
    df["ep_next"] = df["ep_next"].apply(to_float)
    df["ppg"] = pd.to_numeric(df["points_per_game"], errors="coerce")
    df["exp_points_next"] = df["ep_next"].fillna(df["ppg"]).fillna(0.0)
    df["cost_m"] = df["now_cost"] / 10.0

    # Availability filter
    if not allow_flagged:
        thr = 75 if min_play_chance is None else int(min_play_chance)
        # treat NaN as 100 so fully available players never get dropped due to missing chance
        ch = df["chance_of_playing_next_round"].fillna(100)
        df = df[(df["status"] == "a") | (ch >= thr)]

    keep = ["id","name","team_id","team_name","pos_code","cost_m","ppg","exp_points_next"]
    return df[keep].reset_index(drop=True)


In [6]:
FIXTURES_URL = f"{FPL_API_BASE}/fixtures/"

def fetch_fixtures() -> pd.DataFrame:
    r = requests.get(FIXTURES_URL, timeout=30)
    r.raise_for_status()
    return pd.DataFrame(r.json())

def get_next_gw_and_window(bootstrap: dict, horizon: int) -> Tuple[int, List[int]]:
    events = pd.DataFrame(bootstrap["events"])
    # Prefer is_next; fallback to the smallest unfinished/upcoming
    if "is_next" in events.columns and events["is_next"].any():
        next_gw = int(events.loc[events["is_next"], "id"].iloc[0])
    else:
        unfinished = events.loc[~events.get("finished", False), "id"]
        next_gw = int(unfinished.min()) if len(unfinished) else int(events["id"].max())
    window = list(range(next_gw, next_gw + horizon))
    return next_gw, window

def compute_team_gw_weights(bootstrap: dict, fixtures_df: pd.DataFrame,
                            gw_window: List[int], diff_to_mult: Dict[int, float]) -> Dict[Tuple[int,int], float]:
    """
    Return weights[(team_id, gw)] = sum of multipliers for that team's fixtures in that GW.
    Handles blanks (no fixtures -> 0.0) and doubles (sums multiples).
    """
    weights = {}
    # Work only on requested GWs
    fx = fixtures_df[fixtures_df["event"].isin(gw_window)].copy()

    # Some fixtures may lack 'event' (postponed). Ignore those for now.
    fx = fx.dropna(subset=["event"])
    fx["event"] = fx["event"].astype(int)

    for _, row in fx.iterrows():
        gw = int(row["event"])
        # Home side
        t_h = int(row["team_h"]); diff_h = int(row.get("team_h_difficulty", 3))
        w_h = diff_to_mult.get(diff_h, 1.0)
        weights[(t_h, gw)] = weights.get((t_h, gw), 0.0) + w_h
        # Away side
        t_a = int(row["team_a"]); diff_a = int(row.get("team_a_difficulty", 3))
        w_a = diff_to_mult.get(diff_a, 1.0)
        weights[(t_a, gw)] = weights.get((t_a, gw), 0.0) + w_a

    # Ensure every (team, gw) in window has at least 0 weight
    teams_df = pd.DataFrame(bootstrap["teams"])
    for team_id in teams_df["id"].tolist():
        for gw in gw_window:
            weights.setdefault((int(team_id), int(gw)), 0.0)
    return weights


In [7]:
## new change implemented 26 september

##Drop-in replacement (keeps the same name/signature, ignores use_fdr_for_gw1)

In [8]:
def add_horizon_expected_points(players_df: pd.DataFrame,
                                bootstrap: dict,
                                fixtures_df: pd.DataFrame,
                                horizon: int,
                                diff_to_mult: Dict[int, float],
                                use_fdr_for_gw1: bool) -> pd.DataFrame:
    """
    Uniform version: uses PPG × fixture multipliers for *all* GWs in the window.
    Doubles add both fixtures; blanks contribute 0.
    Notes:
      - `use_fdr_for_gw1` is accepted for backward compatibility but ignored.
      - If PPG is missing, fall back to exp_points_next, else 0.0.
    """
    next_gw, window = get_next_gw_and_window(bootstrap, horizon)
    weights = compute_team_gw_weights(bootstrap, fixtures_df, window, diff_to_mult)

    def horizon_sum(row):
        team_id = int(row["team_id"])
        # Robust base: prefer PPG; if missing, fall back to exp_points_next; else 0.
        base = float(row["ppg"]) if pd.notnull(row["ppg"]) else (
               float(row["exp_points_next"]) if pd.notnull(row["exp_points_next"]) else 0.0)
        total = 0.0
        for gw in window:
            w = float(weights.get((team_id, int(gw)), 0.0))
            total += base * w
        return total

    df = players_df.copy()
    df["exp_points_h"] = df.apply(horizon_sum, axis=1)
    df.attrs["next_gw"] = next_gw
    df.attrs["gw_window"] = window
    return df


In [9]:
# Fetch fresh
bootstrap = fetch_bootstrap()
fixtures_df = fetch_fixtures()

# Build players with robust mapping
# players = build_player_table(bootstrap, allow_flagged=ALLOW_FLAGGED)
players = build_player_table(bootstrap, allow_flagged=False, min_play_chance=0)  # includes i/s

# Add horizon-weighted expected points
players_h = add_horizon_expected_points(
    players, bootstrap, fixtures_df,
    horizon=HORIZON_GWS,
    diff_to_mult=DIFF_TO_MULT,
    use_fdr_for_gw1=USE_FDR_FOR_GW1,
)






In [10]:
# right after you compute exp_points_h in add_horizon_expected_points, or before optimisation:
AVAIL_SCALE = {
    "a": 1.00,  # available
    "d": 0.85,  # doubtful
    "i": 0.50,  # injured
    "s": 0.40,  # suspended
    "u": 0.60,  # unavailable/other
}
players_h = players_h.merge(
    pd.DataFrame(bootstrap["elements"])[["id","status"]],
    on="id", how="left"
)
players_h["avail_mult"] = players_h["status"].map(AVAIL_SCALE).fillna(0.8)
players_h["exp_points_h"] = players_h["exp_points_h"] * players_h["avail_mult"]


In [11]:
# Sanity check
print("Next GW:", players_h.attrs.get("next_gw"))
print("GW window:", players_h.attrs.get("gw_window"))
players_h[["name","team_name","pos_code","cost_m","exp_points_next","ppg","exp_points_h"]].head(10)

Next GW: None
GW window: None


,name,team_name,pos_code,cost_m,exp_points_next,ppg,exp_points_h
0,Raya,Arsenal,GK,5.6,3.8,4.7,19.74
1,Arrizabalaga,Arsenal,GK,4.3,0.5,0.0,0.00
2,Hein,Arsenal,GK,4.0,0.0,0.0,0.00
3,Setford,Arsenal,GK,4.0,0.5,0.0,0.00
4,Gabriel,Arsenal,DEF,6.2,8.5,6.3,26.46
5,Saliba,Arsenal,DEF,6.0,1.5,3.8,15.96
6,Calafiori,Arsenal,DEF,5.7,3.2,6.0,25.20
7,J.Timber,Arsenal,DEF,5.8,4.2,6.2,26.04
8,Kiwior,Arsenal,DEF,5.4,0.0,0.0,0.00
9,Lewis-Skelly,Arsenal,DEF,5.3,1.2,0.8,3.36


In [12]:
def optimise_squad(
    players: pd.DataFrame,
    budget_m: float = BUDGET_M,
    max_per_team: int = MAX_PER_TEAM,
    required_counts: Dict[str,int] = REQUIRED_COUNTS,
    lock_names: Optional[Set[str]] = None,
    ban_names: Optional[Set[str]] = None,
    min_from_team: Optional[Dict[str,int]] = None,
    verbose: bool = False,
):
    def _build_and_solve(df, budget, max_team, req_counts, locks, bans, min_team, loosen_positions=False):
        # normalise bans
        bans = {b.strip().lower() for b in (bans or set())}
        if bans:
            mask = df["name"].str.lower().apply(lambda nm: any(b in nm for b in bans))
            df = df[~mask].copy()

        model = pulp.LpProblem("FPL_Squad_Optimisation", sense=pulp.LpMaximize)
        idxs = list(df.index)
        x = {i: pulp.LpVariable(f"x_{i}", lowBound=0, upBound=1, cat=pulp.LpBinary) for i in idxs}

        # objective
        model += pulp.lpSum(df.loc[i,"exp_points"] * x[i] for i in idxs)

        # constraints
        model += pulp.lpSum(df.loc[i,"cost_m"] * x[i] for i in idxs) <= budget, "Budget"
        model += pulp.lpSum(x[i] for i in idxs) == 15, "SquadSize"

        for pos, req in req_counts.items():
            pos_idx = df.index[df["pos_code"] == pos]
            if loosen_positions:
                model += pulp.lpSum(x[i] for i in pos_idx) >= req, f"MinCount_{pos}"
            else:
                model += pulp.lpSum(x[i] for i in pos_idx) == req, f"Count_{pos}"

        for team in sorted(df["team_name"].unique()):
            t_idx = df.index[df["team_name"] == team]
            model += pulp.lpSum(x[i] for i in t_idx) <= max_team, f"Max3_{team}"

        # locks (case-insensitive substring)
        locks = {l.lower() for l in (locks or set())}
        for ln in locks:
            matches = df.index[df["name"].str.lower().str.contains(ln)]
            if len(matches) == 0:
                print(f"[warn] Lock '{ln}' matched no players")
            else:
                model += pulp.lpSum(x[i] for i in matches) >= 1, f"Lock_{ln}"

        # min from team
        if min_team:
            for team, k in min_team.items():
                t_idx = df.index[df["team_name"].str.lower() == team.lower()]
                if len(t_idx) > 0:
                    model += pulp.lpSum(x[i] for i in t_idx) >= int(k), f"Min_{team}"

        status = model.solve(pulp.PULP_CBC_CMD(msg=verbose))
        return pulp.LpStatus[status], df, x, model

    tries = []
    tries.append(("strict", players.copy(), budget_m, max_per_team, required_counts, lock_names or set(), ban_names or set(), min_from_team or {}, False))
    if not ALLOW_FLAGGED:
        try:
            relaxed_players = build_player_table(bootstrap, allow_flagged=True)
        except Exception:
            relaxed_players = players.copy()
        tries.append(("allow_flagged", relaxed_players, budget_m, max_per_team, required_counts, lock_names or set(), ban_names or set(), min_from_team or {}, False))
    tries.append(("no_locks", players.copy(), budget_m, max_per_team, required_counts, set(), ban_names or set(), min_from_team or {}, False))
    tries.append(("no_min_team", players.copy(), budget_m, max_per_team, required_counts, lock_names or set(), ban_names or set(), {}, False))
    tries.append(("team_cap_4", players.copy(), budget_m, 4, required_counts, set(), set(), {}, False))
    relaxed_req = {k: v for k, v in required_counts.items()}
    tries.append(("pos_loosen", players.copy(), budget_m, max_per_team, relaxed_req, set(), set(), {}, True))

    for tag, df, bud, mteam, reqs, locks, bans, mint, loosen in tries:
        status, df_used, x, model = _build_and_solve(df, bud, mteam, reqs, locks, bans, mint, loosen_positions=loosen)
        if status == "Optimal":
            sel_mask = [bool(x[i].value()) for i in df_used.index]
            sel = df_used.loc[sel_mask].copy()
            sel_cost = float(sel["cost_m"].sum())
            sel_points = float(sel["exp_points"].sum())
            print(f"[info] Found optimal solution with strategy: {tag}")
            return sel.reset_index(drop=True), sel_cost, sel_points

    raise RuntimeError("Still infeasible after progressive relaxations — check diagnostics and constraints.")


In [13]:
players_for_opt = players_h.rename(columns={"exp_points_h": "exp_points"})
selected, total_cost, total_points = optimise_squad(
    players_for_opt,
    budget_m=BUDGET_M,
    max_per_team=MAX_PER_TEAM,
    required_counts=REQUIRED_COUNTS,
    lock_names=LOCK,
    ban_names=BAN,
    min_from_team=MIN_FROM_TEAM,
    verbose=VERBOSE_SOLVER,
)


[info] Found optimal solution with strategy: strict


In [14]:
sel_sorted = selected.sort_values(["pos_code","team_name","name"]).reset_index(drop=True)
show = sel_sorted.copy()
show["cost_m"] = show["cost_m"].map(lambda x: f"£{x:0.1f}m")
# If you used Option 1, 'exp_points' already is the horizon sum
col = "exp_points" if "exp_points" in show.columns else "exp_points_h"
show[col] = show[col].map(lambda x: f"{x:0.2f}")

from IPython.display import display
display(show)

print("\nTotals over horizon:")
print(f"  Cost: £{total_cost:0.1f}m / {BUDGET_M:0.1f}m")
print(f"  Horizon expected points (sum): {total_points:0.2f} over GWs {players_h.attrs.get('gw_window')}")


,id,name,team_id,team_name,pos_code,cost_m,ppg,exp_points_next,exp_points,status,avail_mult
0,5,Gabriel,1,Arsenal,DEF,£6.2m,6.3,8.5,26.46,a,1.0
1,8,J.Timber,1,Arsenal,DEF,£5.8m,6.2,4.2,26.04,a,1.0
2,72,Senesi,4,Bournemouth,DEF,£4.9m,7.3,6.3,28.10,a,1.0
3,260,Guéhi,8,Crystal Palace,DEF,£4.8m,7.2,6.7,27.72,a,1.0
4,683,Alderete,17,Sunderland,DEF,£4.1m,6.5,9.5,26.33,a,1.0
5,249,João Pedro,7,Chelsea,FWD,£7.7m,6.2,3.2,24.49,a,1.0
6,430,Haaland,13,Man City,FWD,£14.4m,10.3,13.2,40.69,a,1.0
7,714,Woltemade,15,Newcastle,FWD,£7.1m,6.0,6.5,24.60,a,1.0
8,469,Pope,15,Newcastle,GK,£5.1m,5.7,5.8,23.37,a,1.0
9,670,Roefs,17,Sunderland,GK,£4.6m,6.5,6.8,26.33,a,1.0



Totals over horizon:
  Cost: £96.9m / 100.0m
  Horizon expected points (sum): 410.73 over GWs None


In [15]:
def choose_starting_xi(squad_df: pd.DataFrame):
    df = squad_df.copy()
    model = pulp.LpProblem("FPL_Starting_XI", sense=pulp.LpMaximize)

    idxs = list(df.index)
    y = {i: pulp.LpVariable(f"y_{i}", lowBound=0, upBound=1, cat=pulp.LpBinary) for i in idxs}

    # Maximise expected points of XI
    model += pulp.lpSum(df.loc[i,"exp_points"] * y[i] for i in idxs)

    # Exactly 11 players
    model += pulp.lpSum(y[i] for i in idxs) == 11, "XI_size"

    # Position constraints
    def idx_pos(p): return df.index[df["pos_code"] == p]
    model += pulp.lpSum(y[i] for i in idx_pos("GK")) == 1,  "XI_1_GK"
    model += pulp.lpSum(y[i] for i in idx_pos("DEF")) >= 3, "XI_min_DEF_3"
    model += pulp.lpSum(y[i] for i in idx_pos("MID")) >= 2, "XI_min_MID_2"
    model += pulp.lpSum(y[i] for i in idx_pos("FWD")) >= 1, "XI_min_FWD_1"

    status = model.solve(pulp.PULP_CBC_CMD(msg=False))
    if pulp.LpStatus[status] != "Optimal":
        raise RuntimeError(f"XI Solver did not find an optimal solution: {pulp.LpStatus[status]}")

    start = df[[bool(v.value()) for v in y.values()]].copy()
    bench = df[[not bool(v.value()) for v in y.values()]].copy()

    return start.reset_index(drop=True), bench.reset_index(drop=True), float(start["exp_points"].sum())

starting_xi, bench, xi_points = choose_starting_xi(selected)
print("Starting XI (max expected points):")
display(starting_xi.sort_values(["pos_code","team_name","name"]).reset_index(drop=True))
print(f"\nXI expected points: {xi_points:0.2f}")

print("\nBench (unordered):")
display(bench.sort_values(["pos_code","team_name","name"]).reset_index(drop=True))


Starting XI (max expected points):


,id,name,team_id,team_name,pos_code,cost_m,ppg,exp_points_next,exp_points,status,avail_mult
0,5,Gabriel,1,Arsenal,DEF,6.2,6.3,8.5,26.460,a,1.0
1,8,J.Timber,1,Arsenal,DEF,5.8,6.2,4.2,26.040,a,1.0
2,72,Senesi,4,Bournemouth,DEF,4.9,7.3,6.3,28.105,a,1.0
3,260,Guéhi,8,Crystal Palace,DEF,4.8,7.2,6.7,27.720,a,1.0
4,683,Alderete,17,Sunderland,DEF,4.1,6.5,9.5,26.325,a,1.0
5,430,Haaland,13,Man City,FWD,14.4,10.3,13.2,40.685,a,1.0
6,670,Roefs,17,Sunderland,GK,4.6,6.5,6.8,26.325,a,1.0
7,82,Semenyo,4,Bournemouth,MID,7.8,8.0,8.3,30.800,a,1.0
8,237,Enzo,7,Chelsea,MID,6.7,6.5,4.2,25.675,a,1.0
9,267,Sarr,8,Crystal Palace,MID,6.4,7.2,3.7,27.720,a,1.0



XI expected points: 312.80

Bench (unordered):


,id,name,team_id,team_name,pos_code,cost_m,ppg,exp_points_next,exp_points,status,avail_mult
0,249,João Pedro,7,Chelsea,FWD,7.7,6.2,3.2,24.49,a,1.0
1,714,Woltemade,15,Newcastle,FWD,7.1,6.0,6.5,24.60,a,1.0
2,469,Pope,15,Newcastle,GK,5.1,5.7,5.8,23.37,a,1.0
3,200,Anthony,3,Burnley,MID,5.7,6.7,5.8,25.46,a,1.0


In [16]:
# ====== YOUR SQUAD (from your message) ======
CURRENT_SQUAD = [
    # GK
    "Kelleher", "Raya",
    # DEF
    "Cucurella", "Gabriel", "Keane", "Chalobah", "Van de Ven",
    # MID
    "Semenyo", "Mbeumo", "Grealish", "McNeil", "Kevin",
    # FWD
    "Haaland", "Chris Wood", "Richarlison",]

# Money in the bank (approx; used with API 'now_cost'; adjust if you know it)
BANK_M = 0.0


In [17]:
import difflib

def _norm(s): 
    return s.strip().lower().replace("-", " ")

def _best_match(pool_names, q, min_ratio=0.85):
    qn = _norm(q)
    # exact
    for pn in pool_names:
        if _norm(pn) == qn:
            return pn
    # token overlap (keep it permissive for names like "Richarlison de Andrade")
    qtokens = [t for t in qn.split() if t]
    if qtokens:
        cands = [pn for pn in pool_names if any(t in _norm(pn) for t in qtokens)]
        if cands:
            return sorted(cands, key=len)[0]
    # strict fuzzy
    close = difflib.get_close_matches(q, pool_names, n=1, cutoff=min_ratio)
    return close[0] if close else None


def map_names_to_pool(pool_df, names):
    pool_names = pool_df["name"].tolist()
    rows = []
    for nm in names:
        best = _best_match(pool_names, nm)
        if best is None:
            print(f"[warn] Could not match: {nm}")
            continue
        rows.append(pool_df[pool_df["name"] == best].iloc[0])
    return pd.DataFrame(rows).reset_index(drop=True)

current_df = map_names_to_pool(players_h, CURRENT_SQUAD)
print(f"Mapped {len(current_df)}/{len(CURRENT_SQUAD)} players.")
display(current_df[["name","team_name","pos_code","cost_m","exp_points_next","ppg","exp_points_h"]])


Mapped 15/15 players.


,name,team_name,pos_code,cost_m,exp_points_next,ppg,exp_points_h
0,Kelleher,Brentford,GK,4.5,3.2,3.0,11.400
1,Raya,Arsenal,GK,5.6,3.8,4.7,19.740
2,Cucurella,Chelsea,DEF,6.1,-0.2,3.3,13.035
3,Gabriel,Arsenal,DEF,6.2,8.5,6.3,26.460
4,Keane,Everton,DEF,4.5,6.0,5.2,20.540
5,Chalobah,Chelsea,DEF,5.2,0.0,5.8,9.164
6,Van de Ven,Spurs,DEF,4.8,6.2,5.0,20.500
7,Semenyo,Bournemouth,MID,7.8,8.3,8.0,30.800
8,Mbeumo,Man Utd,MID,8.1,2.8,3.5,13.300
9,Grealish,Everton,MID,6.9,1.7,4.5,17.775


In [18]:
def choose_starting_xi_by(players_df: pd.DataFrame, score_col="exp_points_h"):
    df = players_df.copy()
    model = pulp.LpProblem("XI", sense=pulp.LpMaximize)
    idxs = list(df.index)
    y = {i: pulp.LpVariable(f"y_{i}", 0, 1, cat=pulp.LpBinary) for i in idxs}

    model += pulp.lpSum(df.loc[i, score_col] * y[i] for i in idxs)
    model += pulp.lpSum(y[i] for i in idxs) == 11
    def idx_pos(p): return df.index[df["pos_code"] == p]
    model += pulp.lpSum(y[i] for i in idx_pos("GK")) == 1
    model += pulp.lpSum(y[i] for i in idx_pos("DEF")) >= 3
    model += pulp.lpSum(y[i] for i in idx_pos("MID")) >= 2
    model += pulp.lpSum(y[i] for i in idx_pos("FWD")) >= 1

    status = model.solve(pulp.PULP_CBC_CMD(msg=False))
    assert pulp.LpStatus[status] == "Optimal"

    xi = df[[bool(v.value()) for v in y.values()]]
    bench = df[[not bool(v.value()) for v in y.values()]]
    return xi.sort_values(["pos_code","team_name","name"]), bench, float(xi[score_col].sum())

xi, bench, xi_pts = choose_starting_xi_by(current_df, "exp_points_h")
print(f"Best XI (horizon points): {xi_pts:.2f}")
display(xi[["name","team_name","pos_code","cost_m","exp_points_h"]])
print("\nBench (unordered):")
display(bench[["name","team_name","pos_code","cost_m","exp_points_h"]])


Best XI (horizon points): 238.40


,name,team_name,pos_code,cost_m,exp_points_h
3,Gabriel,Arsenal,DEF,6.2,26.460
2,Cucurella,Chelsea,DEF,6.1,13.035
4,Keane,Everton,DEF,4.5,20.540
6,Van de Ven,Spurs,DEF,4.8,20.500
12,Haaland,Man City,FWD,14.4,40.685
13,Wood,Nott'm Forest,FWD,7.4,14.245
14,Richarlison,Spurs,FWD,6.8,21.320
1,Raya,Arsenal,GK,5.6,19.740
7,Semenyo,Bournemouth,MID,7.8,30.800
9,Grealish,Everton,MID,6.9,17.775



Bench (unordered):


,name,team_name,pos_code,cost_m,exp_points_h
0,Kelleher,Brentford,GK,4.5,11.400
5,Chalobah,Chelsea,DEF,5.2,9.164
10,McNeil,Everton,MID,5.6,3.950
11,Kevin,Fulham,MID,6.0,2.660


In [19]:
def team_counts(df):
    return df["team_name"].value_counts().to_dict()

def max_three_ok(after_df):
    return (after_df["team_name"].value_counts() <= 3).all()

def affordable_after_swap(current_df, out_row, in_row, bank_m):
    current_cost = float(current_df["cost_m"].sum())
    new_cost = current_cost - float(out_row["cost_m"]) + float(in_row["cost_m"])
    return new_cost <= current_cost + bank_m + 1e-9

def apply_swap(current_df, out_idx, in_row):
    new_df = current_df.copy()
    new_df.loc[out_idx] = in_row
    return new_df

def shortlist_in_pool(pool, pos, exclude_names, max_price=None):
    df = pool.copy()
    df = df[df["pos_code"] == pos]
    if exclude_names:
        ex = set(n.lower() for n in exclude_names)
        df = df[~df["name"].str.lower().isin(ex)]
    if max_price is not None:
        df = df[df["cost_m"] <= max_price + 1e-9]
    # sort by horizon points
    return df.sort_values("exp_points_h", ascending=False)




def best_single_transfer(players_h, current_df, bank_m=0.0, top_k=10):
    base_xi_pts = choose_starting_xi_by(current_df, "exp_points_h")[2]
    already = set(current_df["name"].str.lower().tolist())
    results = []

    # precompute team counts
    base_team_counts = team_counts(current_df)

    for out_idx, out_row in current_df.reset_index().iterrows():
        pos = out_row["pos_code"]
        # Only same-position ins to keep the 15-man shape stable
        candidate_ins = shortlist_in_pool(players_h, pos, exclude_names=already, max_price=out_row["cost_m"] + bank_m)
        for _, in_row in candidate_ins.iterrows():
            # Budget check (approx, using now_cost)
            if not affordable_after_swap(current_df, out_row, in_row, bank_m):
                continue
            # Team cap check (<=3 per club)
            after = apply_swap(current_df, out_idx, in_row)
            if not max_three_ok(after):
                continue
            # Recompute XI points
            new_xi_pts = choose_starting_xi_by(after, "exp_points_h")[2]
            delta = new_xi_pts - base_xi_pts
            if delta > 0.01:
                results.append({
                    "out": out_row["name"], "out_team": out_row["team_name"], "out_pos": pos, "out_cost": float(out_row["cost_m"]),
                    "in": in_row["name"], "in_team": in_row["team_name"], "in_pos": in_row["pos_code"], "in_cost": float(in_row["cost_m"]),
                    "delta_pts": float(delta)
                })
    if not results:
        return pd.DataFrame(columns=["out","out_team","out_pos","out_cost","in","in_team","in_pos","in_cost","delta_pts"])
    recs = pd.DataFrame(results).sort_values("delta_pts", ascending=False).head(top_k)
    return recs.reset_index(drop=True)

recs = best_single_transfer(players_h, current_df, bank_m=BANK_M, top_k=12)
print("Top single-transfer gains (horizon points):")
display(recs)


Top single-transfer gains (horizon points):


,out,out_team,out_pos,out_cost,in,in_team,in_pos,in_cost,delta_pts
0,Cucurella,Chelsea,DEF,6.1,Senesi,Bournemouth,DEF,4.9,15.070
1,Chalobah,Chelsea,DEF,5.2,Senesi,Bournemouth,DEF,4.9,15.070
2,Cucurella,Chelsea,DEF,6.1,Guéhi,Crystal Palace,DEF,4.8,14.685
3,Chalobah,Chelsea,DEF,5.2,Guéhi,Crystal Palace,DEF,4.8,14.685
4,Mbeumo,Man Utd,MID,8.1,Sarr,Crystal Palace,MID,6.4,14.420
5,Kevin,Fulham,MID,6.0,Gravenberch,Liverpool,MID,5.6,13.915
6,McNeil,Everton,MID,5.6,Gravenberch,Liverpool,MID,5.6,13.915
7,Mbeumo,Man Utd,MID,8.1,Gravenberch,Liverpool,MID,5.6,13.650
8,Cucurella,Chelsea,DEF,6.1,Alderete,Sunderland,DEF,4.1,13.290
9,Chalobah,Chelsea,DEF,5.2,Alderete,Sunderland,DEF,4.1,13.290
